# D1.1 · From alert queue to loop operator

**Function D — The Agentic SOC → The Agentic SOC — Detection**  ·  *AI for Security*

Builds on **[D1.0 · Start here — what AI for security operations means](https://spbreed.github.io/cyber-commons/lessons/D1.0.html)**.

| | |
|---|---|
| Tools used | Wazuh, OpenSearch, GLM-4.6, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Run a triage loop over Wazuh alerts and supervise by exception.

**Why a security engineer needs it.** Supervising by re-reading everything the loop did. The control it builds is: know what the loop must escalate and sample the rest.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

The queue does not go away; it changes shape. Instead of triaging alerts you are supervising something that triages alerts, which is a different skill with a different quality bar and a much worse failure mode: confident, fast, and wrong at volume.

> **At CyberTravels.** The analyst on CyberTravels' alerts stops triaging and starts supervising something that triages — which is a different skill, with a worse failure mode: confident, fast, and wrong at volume.

## 2 · The framework

```
   before                          after
   +------------------+            +---------------------------+
   | alert -> analyst |            | alert -> loop -> analyst  |
   |          decides |            |          proposes  reviews|
   +------------------+            +---------------------------+
      100 alerts/day                  1000 alerts/day, 40 reviewed

   new failure mode: confident, fast, and wrong at volume
```

The classic SOC job is a queue: alerts arrive, an analyst reads each one,
decides, and moves on. The constraint is human attention, and it does not scale
— which is why tier-1 burnout and alert fatigue are structural rather than
cultural problems.

The agentic version replaces "read every alert" with "operate a loop that reads
every alert". The analyst's job becomes:

- deciding **what the loop is allowed to conclude** (the verifier, B2.0),
- deciding **what it may do about it** (the tool policy, A3.5),
- and handling the cases it escalates.

The skill that transfers is not triage speed. It is knowing which signals the
loop may believe — because a triage loop with a weak verifier closes true
positives at machine speed, and closing a true positive is silent.

## 3 · The model backend, and the disposition it proposes

In [ ]:
# --- model backend: replay by default, a Kaggle open-weight model when served -
# One URL and one header shape, no vendor SDK. Standard library only, so the
# notebook stays self-contained.
import json, os, urllib.error, urllib.request

# Qwen2.5-7B-Instruct is the floor established in MODELS.md: below it two of
# the lessons' acceptance properties stop holding.
OPEN_WEIGHT_DEFAULT = "qwen2.5-7b-instruct"
TIMEOUT = 60

def backend():
    """(kind, model). Configuration comes from the environment, never a literal."""
    if os.environ.get("OPENAI_BASE_URL"):
        return "open-weight", os.environ.get("MODEL", OPEN_WEIGHT_DEFAULT)
    return "replay", "deterministic stand-in (no backend configured)"

def _post(url, payload, headers):
    req = urllib.request.Request(url, data=json.dumps(payload).encode(),
                                 headers={"content-type": "application/json", **headers})
    with urllib.request.urlopen(req, timeout=TIMEOUT) as r:
        return json.loads(r.read().decode())

def _openai_compatible(prompt, system, model, max_tokens, temperature):
    msgs = ([{"role": "system", "content": system}] if system else []) + \
           [{"role": "user", "content": prompt}]
    base = os.environ["OPENAI_BASE_URL"].rstrip("/")
    key = os.environ.get("OPENAI_API_KEY", "not-needed")
    out = _post(f"{base}/chat/completions",
                {"model": model, "messages": msgs, "max_tokens": max_tokens,
                 "temperature": temperature},
                {"authorization": f"Bearer {key}"})
    return out["choices"][0]["message"]["content"].strip()

def ask(prompt, *, replay, system=None, max_tokens=512, temperature=0.0):
    """Answer `prompt` with the configured backend, or return `replay`.

    `replay` is required, not optional: a lesson must be able to run offline,
    and the answer it falls back to has to be visible in the source rather than
    invented at runtime.
    """
    kind, model = backend()
    if kind == "replay":
        return replay, kind, model
    try:
        return _openai_compatible(prompt, system, model, max_tokens,
                                  temperature), kind, model
    except (urllib.error.URLError, urllib.error.HTTPError, KeyError, TimeoutError) as e:
        # Print what the server actually said. "failed: 400" costs whoever hits
        # this an hour; the body usually names the exact missing parameter, and
        # it never contains a key.
        detail = getattr(e, "code", None) or type(e).__name__
        why = ""
        if hasattr(e, "read"):
            try:
                why = json.loads(e.read().decode()).get("error", {}).get("message", "")
            except Exception:
                why = ""
        print(f"   !! {kind} backend ({model}) failed: {detail}"
              f"{' - ' + why if why else ''}")
        print("      Using the replay, which is labelled as one. No model answered.")
        return replay, "replay", f"{model} unreachable"

_kind, _model = backend()
print(f"model backend : {_kind}")
print(f"model         : {_model}")
if _kind == "replay":
    print()
    print("This lesson runs offline against a deterministic replay, which is why")
    print("it works on a Kaggle kernel with the internet switched off. To run the")
    print("identical code against a real model, serve an open-weight model from")
    print("Kaggle Models and point the adapter at it:")
    print()
    print("   python3 -m llama_cpp.server --model <the .gguf from Kaggle> \\")
    print("           --model_alias qwen2.5-7b-instruct --port 11434 --chat_format qwen")
    print("   export OPENAI_BASE_URL=http://127.0.0.1:11434/v1 \\")
    print("          MODEL=qwen2.5-7b-instruct")
    print()
    print("   MODELS.md has the exact Kaggle download. There is no paid backend:")
    print("   every model result in this repository was produced this way.")

## 4 · The same lesson, against a real model

Everything below this point runs identically on two backends. Offline it uses a
deterministic replay that is labelled as a replay wherever it appears — never
presented as a model's output. With `OPENAI_BASE_URL` set it calls an
OpenAI-compatible server, which is how the open-weight models on Kaggle are
served — and how every model result in this repository was produced.

The point of running it both ways is not that the answers match. It is that
**the lesson's assertion holds either way** — if it only holds against the
replay, the lesson was testing the replay.

In [ ]:
TASK = 'Triage this alert to one of: escalate, close-benign, needs-context.\n\nAlert: service account svc-reports authenticated from 203.0.113.9 at 03:14 and listed all S3 buckets. svc-reports normally runs hourly from 10.2.0.0/16 and touches one bucket.'

REPLAY = "escalate - the source range and the breadth of the list call are both outside this account's established pattern."

answer, used, model = ask(TASK, replay=REPLAY,
            system='You are a SOC triage assistant. One line: disposition, then why.',
            max_tokens=300)

print(f"backend used : {used}")
print(f"model        : {model}")
print(f"prompt       : {TASK[:66]}...")
print()
print("answer:")
for line in (answer.splitlines() or [answer]):
    print(f"   {line}")

# Two assertions that must hold on every backend, and one property that is
# reported rather than asserted - a real model failing it is a finding about
# the model, not a broken notebook.
assert answer.strip(), "the configured backend returned nothing"
if used == "replay":
    assert answer == REPLAY, "the offline path must return the replay verbatim"

label, held = ("returned one of the three dispositions", any(d in answer.lower() for d in ("escalate", "close-benign", "needs-context")))
print()
print(f"property checked : {label}")
print(f"held on {used:12s} : {held}")
print()
print("Same code, same assertions, two possible backends. Offline the answer is")
print("the replay and is labelled as one; with a served model it is the model's.")

## 5 · Where it breaks — closing a true positive is silent

Every triage decision has two error directions and they are not symmetric. Escalating a false positive costs an analyst ten minutes. **Closing a true positive costs you the incident**, and nothing tells you it happened.

## 6 · The control — the loop may close, but not silently

Three rules make an agentic triage loop safe to run, and none of them is about model quality.

## What you just proved

The triage loop escalates 4 alerts and closes 4, matching ground truth on all 8. Lowering the confidence bar trades analyst minutes against missed incidents. The severity floor converts any high or critical closure into an escalation, and the closure sampling routes a fraction of routine closures to a human for quality measurement.

## Your turn

Ask your SOC one question: when an incident is confirmed, does anyone check whether an earlier alert about it was closed? If nobody does, you have no measurement of your false-negative rate — with or without an agent.

---

**Next → [D1.2 · Context that makes triage work](https://spbreed.github.io/cyber-commons/lessons/D1.2.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D1.1.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D1.1.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*